In [3]:
# Imports

import os
import sys
import numpy as np
import netCDF4 as nc
from tqdm.auto import tqdm # progress bar
import xarray as xr
import xesmf as xe
from cartopy.crs import PlateCarree
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import glob
from matplotlib.colors import ListedColormap, BoundaryNorm
from dask.diagnostics import ProgressBar


In [4]:
def mean_seasonal_cycle(ds, time_dim="time", groupby="time.month"):
    """
    Remove the seasonal mean cycle from an xarray dataset.

    Parameters:
    ds (xr.Dataset or xr.DataArray): Input dataset or data array.
    time_dim (str): Name of the time dimension.
    groupby (str): Grouping method, default is 'time.month' (monthly climatology).

    Returns:
    xr.Dataset or xr.DataArray: Dataset or data array with the seasonal cycle removed.
    """
    climatology = ds.groupby(groupby).mean(dim=time_dim)
    return climatology

def remove_seasonal_cycle(ds, time_dim="time", groupby="time.month"):
    """
    Remove the seasonal mean cycle from an xarray dataset.

    Parameters:
    ds (xr.Dataset or xr.DataArray): Input dataset or data array.
    time_dim (str): Name of the time dimension.
    groupby (str): Grouping method, default is 'time.month' (monthly climatology).

    Returns:
    xr.Dataset or xr.DataArray: Dataset or data array with the seasonal cycle removed.
    """
    climatology = ds.groupby(groupby).mean(dim=time_dim)
    return climatology, ds.groupby(groupby) - climatology

In [5]:
viking_ds = xr.open_dataset("/glade/work/netige/Data/VIKING/obp_ssh_rm_dt.nc")

In [6]:
viking_ds

<xarray.Dataset>
Dimensions:      (time: 372, lon: 2715, lat: 2267)
Coordinates:
  * time         (time) datetime64[ns] 1993-01-16T12:00:00 ... 2023-12-16T12:...
  * lon          (lon) float64 -97.96 -97.91 -97.87 -97.83 ... 19.89 19.93 19.98
  * lat          (lat) float64 -33.48 -33.44 -33.39 -33.35 ... 64.9 64.94 64.99
Data variables:
    obp_rm_dt    (time, lat, lon) float32 ...
    ssh_p_rm_dt  (time, lat, lon) float32 ...
    ssh_rm_dt    (time, lat, lon) float32 ...
Attributes:
    title:        Mean-removed & detrended OBP and SSH variants
    history:      Created with xarray.to_netcdf (chunked + compressed)
    Conventions:  CF-1.8

In [7]:
%%time
[viking_obp_MSC, viking_obp_res] = remove_seasonal_cycle(viking_ds.obp_rm_dt)

CPU times: user 6min 50s, sys: 13 s, total: 7min 3s
Wall time: 8min 27s


In [8]:
%%time
[viking_ssh_MSC, viking_ssh_res] = remove_seasonal_cycle(viking_ds.ssh_rm_dt)

CPU times: user 6min 49s, sys: 12.9 s, total: 7min 2s
Wall time: 8min 39s


In [9]:
print(viking_obp_MSC, viking_obp_res, viking_ssh_MSC, viking_ssh_res)

<xarray.DataArray 'obp_rm_dt' (month: 12, lat: 2267, lon: 2715)>
array([[[-0.16308954, -0.16174942, -0.1660238 , ...,         nan,
                 nan,         nan],
        [-0.1615817 , -0.161271  , -0.16513255, ...,         nan,
                 nan,         nan],
        [-0.15915555, -0.15689552, -0.15917744, ...,         nan,
                 nan,         nan],
        ...,
        [ 0.16245951,  0.16245951,  0.16245951, ...,  0.16245951,
          0.16245951,  0.16245951],
        [ 0.16245951,  0.16245951,  0.16245951, ...,  0.16245951,
          0.16245951,  0.16245951],
        [ 0.16245951,  0.16245951,  0.16245951, ...,  0.16245951,
          0.16245951,  0.16245951]],

       [[ 0.03230278,  0.02757996,  0.02530463, ...,         nan,
                 nan,         nan],
        [ 0.03784367,  0.03108724,  0.03124217, ...,         nan,
                 nan,         nan],
        [ 0.04027161,  0.04050174,  0.03919982, ...,         nan,
                 nan,         nan],
..

In [10]:
import xarray as xr

# Ensure each DataArray has a unique name before putting into a Dataset
ds_out = xr.Dataset(
    data_vars={
        "viking_obp_MSC": viking_obp_MSC.rename("viking_obp_MSC"),
        "viking_obp_res": viking_obp_res.rename("viking_obp_res"),
        "viking_ssh_MSC": viking_ssh_MSC.rename("viking_ssh_MSC"),
        "viking_ssh_res": viking_ssh_res.rename("viking_ssh_res"),
    },
    coords={
        # Reuse coords from one of them (they should be identical on lat/lon)
        "lat": viking_obp_res["lat"],
        "lon": viking_obp_res["lon"],
        "time": viking_obp_res["time"],
        "month": viking_obp_MSC["month"],
    },
    attrs={
        "source": "VIKING",
        "description": "OBP/SSH mean seasonal cycle (MSC; month) and residual (res; time).",
    },
)

# Optional: preserve original variable attrs (units, long_name, etc.)
for v in ds_out.data_vars:
    ds_out[v].attrs = dict(ds_out[v].attrs)  # keep whatever is already there

# If you want compression (strongly recommended for these sizes)
encoding = {v: {"zlib": True, "complevel": 4} for v in ds_out.data_vars}

out_path = "/glade/work/netige/Data/VIKING/viking_obp_ssh_msc_residuals.nc"
with ProgressBar():
    ds_out.to_netcdf(out_path, encoding=encoding)

print("Wrote:", out_path)


Wrote: /glade/work/netige/Data/VIKING/viking_obp_ssh_msc_residuals.nc
